In [ ]:
import gc
import os

os.environ["KERAS_BACKEND"] = "jax"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import keras
import keras_hub
import tensorflow as tf
import kagglehub

# Authenticate to Kaggle
kagglehub.auth.authenticate_as_user()

# Set precision to bfloat16 to reduce memory and speed up training on modern GPUs
keras.config.set_dtype_policy("bfloat16")

# -----------------------------------------------------------------------------
# 1. Creuset Domain Dataset Assembly
# -----------------------------------------------------------------------------
# Formatted for startup funding extraction and financial evaluation tasks.

creuset_raw_data = [
    {
        "query": "Analyze this venture update: PayStack raised $200K seed from Y Combinator with $15k MRR.",
        "analysis": "{\n  \"startup\": \"PayStack\",\n  \"round\": \"Seed\",\n  \"amount_usd\": 200000,\n  \"investors\": [\"Y Combinator\"],\n  \"metrics\": {\"mrr_usd\": 15000}\n}"
    },
    {
        "query": "Calculate runway: Startup HAS $1.2M in cash reserves, spending $80k/mo, bringing in $30k/mo MRR.",
        "analysis": "{\n  \"net_burn_monthly\": 50000,\n  \"cash_reserve\": 1200000,\n  \"runway_months\": 24.0,\n  \"status\": \"HEALTHY\"\n}"
    },
    {
        "query": "Extract entity data: Moniepoint secured $110M Series C led by Development Partners International.",
        "analysis": "{\n  \"startup\": \"Moniepoint\",\n  \"round\": \"Series C\",\n  \"amount_usd\": 110000000,\n  \"investors\": [\"Development Partners International\"],\n  \"metrics\": {}\n}"
    }
]

# Convert dictionary list to TensorFlow Dataset
raw_ds = tf.data.Dataset.from_generator(
    lambda: creuset_raw_data,
    output_signature={
        "query": tf.TensorSpec(shape=(), dtype=tf.string),
        "analysis": tf.TensorSpec(shape=(), dtype=tf.string),
    }
)

# Format dataset using Gemma's instruction-tuning tokens
train_ds = raw_ds.map(
    lambda x: tf.strings.join(
        [
            "<start_of_turn>user\n",
            "You are Creuset Intelligence Copilot. Extract structured startup analytics:\n",
            x["query"],
            "<end_of_turn>\n",
            "<start_of_turn>model\n",
            "Analysis:\n",
            x["analysis"],
            "<end_of_turn>",
        ]
    )
)

# Batch operations for low-memory training footprint
train_ds = train_ds.batch(1)

# -----------------------------------------------------------------------------
# 2. Model Initialization & Int8 QLoRA Setup
# -----------------------------------------------------------------------------

# Load Gemma 1.1 2B Causal LM with sequence length constrained to target window
preprocessor = keras_hub.models.GemmaCausalLMPreprocessor.from_preset(
    "gemma_1.1_instruct_2b_en",
    sequence_length=512
)

creuset_copilot = keras_hub.models.GemmaCausalLM.from_preset(
    "gemma_1.1_instruct_2b_en",
    preprocessor=preprocessor
)

# Step 1: Quantize weights dynamically to int8 (QLoRA Step 1)
creuset_copilot.quantize("int8")

# Step 2: Inject low-rank matrices (rank=4) into linear projections (QLoRA Step 2)
# Drops trainable parameters from ~2.5 Billion to ~1.3 Million
creuset_copilot.backbone.enable_lora(rank=4)

creuset_copilot.summary()

# -----------------------------------------------------------------------------
# 3. Model Compilation & Domain Fine-Tuning
# -----------------------------------------------------------------------------

# Use Stochastic Gradient Descent (SGD) to minimize optimizer state memory allocation
optimizer = keras.optimizers.SGD(learning_rate=2e-4)

creuset_copilot.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
)

# Run fine-tuning epoch across structured venture dataset
creuset_copilot.fit(train_ds, epochs=3)

# -----------------------------------------------------------------------------
# 4. Verification & Structured Inference
# -----------------------------------------------------------------------------

prompt_template = (
    "<start_of_turn>user\n"
    "You are Creuset Intelligence Copilot. Extract structured startup analytics:\n"
    "{user_query}"
    "<end_of_turn>\n"
    "<start_of_turn>model\n"
    "Analysis:\n"
)

eval_input = prompt_template.format(
    user_query="Analyze this venture update: Wave raised $200M Series A led by Sequoia Heritage."
)

output = creuset_copilot.generate(eval_input, max_length=512)
extracted_json = output.replace(eval_input, "")

print("\n--- Creuset Intelligence Inference ---")
print(extracted_json)

# Cleanup resources
del preprocessor
del creuset_copilot
del optimizer
gc.collect()

AttributeError: module 'kagglehub.auth' has no attribute 'authenticate_as_user'